# Homework

Complete five true/false explanations and five coding tasks. Code cells are independent; include required implementations and setup. Sign in before saving answers.

## True or False

```{index} True or False
```

**1. Cache ownership**

Caching only remaining amount is sufficient for one-use invoice search, regardless of which positions remain.

- True
- False

```{dropdown} Answer
False. Position identifies remaining inventory; the same amount can have different answers for different suffixes.
```

**2. Zero target**

An empty coin witness always means the target is unreachable.

- True
- False

```{dropdown} Answer
False. A zero count with an empty witness is the valid zero target; null count denotes unreachable.
```

**3. Signed pruning**

A negative remaining amount proves failure even when later invoices may be negative.

- True
- False

```{dropdown} Answer
False. A later credit can restore the target, as 8 followed by −3 makes five. Signed suffix bounds are safe.
```

**4. Space accounting**

Memoized Fibonacci uses constant auxiliary storage because each subproblem is computed once.

- True
- False

```{dropdown} Answer
False. Its cache and active recursion stack each use linear space; rolling tabulation can keep constant state for a final value.
```

**5. Output and limits**

A budget-limited search may report unknown, and materializing every permutation still requires space for all outputs.

- True
- False

```{dropdown} Answer
True. Budget exhaustion is not proof of infeasibility; storing n! length-n orders requires Θ(n·n!) output space.
```

## Coding Questions

```{index} Coding Questions
```

## 6. Count Stair Plans

```{index} 6. Count Stair Plans
```

A person climbs one or two stairs at a time. Use a bottom-up table with W(0)=1 (the empty plan), W(1)=1, W(n)=W(n−1)+W(n−2). Validate 0..91; check zero and five. Explain why these base cases differ from Fibonacci.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;

int n=5;
// Your code starts here.
/* TODO: complete the task. */
// Your code ends here.

In [1]:
using System;
using System.Linq;
using System.Collections.Generic;

// Solution
int n=5;
long Ways(int count)
{
 if(count<0||count>91)throw new ArgumentOutOfRangeException(nameof(count));
 var ways=new long[count+1];ways[0]=1;if(count>=1)ways[1]=1;
 for(int i=2;i<=count;i++)ways[i]=checked(ways[i-1]+ways[i-2]);
 return ways[count];
}
Console.WriteLine($"zero ways={Ways(0)}, five ways={Ways(n)}, max ways={Ways(91)}");
try{Ways(92);}catch(ArgumentOutOfRangeException){Console.WriteLine("stairs overflow domain rejected");}


zero ways=1, five ways=8, max ways=7540113804746346429
stairs overflow domain rejected


## 7. Different Coins and Unreachable States

```{index} 7. Different Coins and Unreachable States
```

For positive sizes 2,5,5, compute targets zero, one, six, and eleven. Validate reachable witnesses, normalize duplicate denominations, and reject negative target and negative denomination.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;

int[] sizes={2,5,5};
// Your code starts here.
/* TODO: complete the task. */
// Your code ends here.

In [1]:
using System;
using System.Linq;
using System.Collections.Generic;

// Solution
int[] sizes={2,5,5};
foreach(int target in new[]{0,1,6,11})
{
 var p=CoinPlanning.MinimumCoins(sizes,target);
 if(p.Count is not null&&(p.Coins.Length!=p.Count||p.Coins.Sum()!=target))throw new Exception("Witness failed.");
 Console.WriteLine($"target={target}, count={p.Count?.ToString()??"unreachable"}, coins=[{string.Join(",",p.Coins)}]");
}
try{CoinPlanning.MinimumCoins(sizes,-1);}catch(ArgumentOutOfRangeException){Console.WriteLine("negative target rejected");}
try{CoinPlanning.MinimumCoins(new[]{-2,5},6);}catch(ArgumentException){Console.WriteLine("negative coin rejected");}
public static class CoinPlanning
{
    public record CoinPlan(int? Count,int[] Coins,int?[] Best);
    // Unlimited positive denominations. A target cap bounds table allocation.
    public static CoinPlan MinimumCoins(int[] denominations,int target)
    {
        ArgumentNullException.ThrowIfNull(denominations);
        if(target<0||target>1_000_000)throw new ArgumentOutOfRangeException(nameof(target));
        if(denominations.Any(c=>c<=0))throw new ArgumentException("Coins must be positive.");
        int[] coins=denominations.Distinct().OrderBy(c=>c).ToArray();
        var best=new int?[target+1];var last=new int[target+1];best[0]=0;
        for(int amount=1;amount<=target;amount++)
        {
            foreach(int coin in coins)
            {
                if(coin>amount)break;
                if(best[amount-coin] is not int count)continue;
                int candidate=count+1;
                if(best[amount] is null||candidate<best[amount])
                {best[amount]=candidate;last[amount]=coin;}
            }
        }
        var chosen=new List<int>();
        if(best[target] is not null)
            for(int remaining=target;remaining>0;remaining-=last[remaining])chosen.Add(last[remaining]);
        return new CoinPlan(best[target],chosen.ToArray(),best);
    }
}


target=0, count=0, coins=[]
target=1, count=unreachable, coins=[]
target=6, count=3, coins=[2,2,2]
target=11, count=4, coins=[2,2,2,5]
negative target rejected
negative coin rejected


## 8. Preserve Signed and Wide Sums

```{index} 8. Preserve Signed and Wide Sums
```

Find a witness for two int.MaxValue invoices and one −1 credit totaling 4294967293. Check distinct valid positions and long sum. Test impossible long.MinValue and an empty zero target.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;

int[] invoices={int.MaxValue,int.MaxValue,-1};long target=4294967293;
// Your code starts here.
/* TODO: complete the task. */
// Your code ends here.

In [1]:
using System;
using System.Linq;
using System.Collections.Generic;

// Solution
int[] invoices={int.MaxValue,int.MaxValue,-1};long target=4294967293;
var r=SubsetPlanning.FindSubset(invoices,target);
if(r.Indices is null||r.Indices.Distinct().Count()!=r.Indices.Length||r.Indices.Any(i=>i<0||i>=invoices.Length)||r.Indices.Sum(i=>(long)invoices[i])!=target)throw new Exception("Wide witness failed.");
Console.WriteLine($"sum={r.Indices.Sum(i=>(long)invoices[i])}, IDs={string.Join(",",r.Indices)}");
Console.WriteLine($"extreme target impossible={SubsetPlanning.FindSubset(invoices,long.MinValue).Indices is null}, empty zero witness={SubsetPlanning.FindSubset(Array.Empty<int>(),0).Indices!.Length}");
public static class SubsetPlanning
{
    public record SubsetResult(int[]? Indices,int Calls,int Hits,int Prunes,int FailedStates);
    // Signed values are allowed; positions are distinct and each is used once.
    public static SubsetResult FindSubset(int[] values,long target,bool prune=true,bool memoize=true,int maxCalls=1_000_000)
    {
        ArgumentNullException.ThrowIfNull(values);
        if(values.Length>128)throw new ArgumentException("At most 128 positions in this teaching implementation.");
        if(maxCalls<1)throw new ArgumentOutOfRangeException(nameof(maxCalls));
        int[] data=(int[])values.Clone();int n=data.Length;
        var lower=new long[n+1];var upper=new long[n+1];
        for(int i=n-1;i>=0;i--)
        {lower[i]=lower[i+1]+Math.Min(0,data[i]);upper[i]=upper[i+1]+Math.Max(0,data[i]);}
        var failed=new HashSet<(int Index,long Remaining)>();var chosen=new List<int>();
        int calls=0,hits=0,prunes=0;int[]? answer=null;
        bool Search(int index,long remaining)
        {
            if(calls==maxCalls)throw new InvalidOperationException("Search budget exhausted; feasibility is unknown.");
            calls++;
            if(remaining==0){answer=chosen.ToArray();return true;}
            if(index==n)return false;
            if(prune&&(remaining<lower[index]||remaining>upper[index])){prunes++;return false;}
            var state=(index,remaining);
            if(memoize&&failed.Contains(state)){hits++;return false;}
            chosen.Add(index);bool found;
            try{found=Search(index+1,remaining-data[index]);}
            finally{chosen.RemoveAt(chosen.Count-1);}
            if(found||Search(index+1,remaining))return true;
            if(memoize)failed.Add(state);
            return false;
        }
        // Even an unpruned trace rejects targets outside the total signed range.
        if(target<lower[0]||target>upper[0]){calls=1;prunes=1;}
        else Search(0,target);
        if(chosen.Count!=0)throw new InvalidOperationException("Undo failed.");
        return new SubsetResult(answer,calls,hits,prunes,failed.Count);
    }
}


sum=4294967293, IDs=0,1,2
extreme target impossible=True, empty zero witness=0


## 9. Enumeration Contracts

```{index} 9. Enumeration Contracts
```

Generate subsets and permutations of two records both labeled Audit. Keep their positions distinct, check cardinalities and unique position outputs, and show the empty-input result count. Explain why distinct label strings are fewer than position permutations.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;

string[] labels={"Audit","Audit"};
// Your code starts here.
/* TODO: complete the task. */
// Your code ends here.

In [1]:
using System;
using System.Linq;
using System.Collections.Generic;

// Solution
string[] labels={"Audit","Audit"};
var subsets=ChoicePlanning.Subsets(labels.Length);var orders=ChoicePlanning.Permutations(labels.Length);
if(subsets.Length!=4||orders.Length!=2||orders.Select(o=>string.Join(",",o)).Distinct().Count()!=2)throw new Exception("Enumeration failed.");
Console.WriteLine($"subsets={subsets.Length}, position orders={orders.Length}, label orders={orders.Select(o=>string.Join("/",o.Select(i=>labels[i]))).Distinct().Count()}");
Console.WriteLine($"empty subsets={ChoicePlanning.Subsets(0).Length}, empty permutations={ChoicePlanning.Permutations(0).Length}");
public static class ChoicePlanning
{
    public static int[][] Subsets(int n)
    {
        if(n<0||n>16)throw new ArgumentOutOfRangeException(nameof(n));
        var result=new List<int[]>();var chosen=new List<int>();
        void Visit(int index)
        {
            if(index==n){result.Add(chosen.ToArray());return;}
            chosen.Add(index);Visit(index+1);chosen.RemoveAt(chosen.Count-1);
            Visit(index+1);
        }
        Visit(0);return result.ToArray();
    }
    public static int[][] Permutations(int n)
    {
        if(n<0||n>8)throw new ArgumentOutOfRangeException(nameof(n));
        var result=new List<int[]>();var chosen=new List<int>();var used=new bool[n];
        void Visit()
        {
            if(chosen.Count==n){result.Add(chosen.ToArray());return;}
            for(int i=0;i<n;i++)
            {
                if(used[i])continue;
                used[i]=true;chosen.Add(i);Visit();chosen.RemoveAt(chosen.Count-1);used[i]=false;
            }
        }
        Visit();return result.ToArray();
    }
}


subsets=4, position orders=2, label orders=1
empty subsets=1, empty permutations=1


## 10. Resource Limits and Honest Conclusions

```{index} 10. Resource Limits and Honest Conclusions
```

Trigger a one-call search budget on values 2,2,2 and target three; catch the budget exception as unknown. Run the same input with its normal budget and check proven infeasibility. Explain why catch-and-return-false would be incorrect.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;

int[] values={2,2,2};long target=3;
// Your code starts here.
/* TODO: complete the task. */
// Your code ends here.

In [1]:
using System;
using System.Linq;
using System.Collections.Generic;

// Solution
int[] values={2,2,2};long target=3;
try{SubsetPlanning.FindSubset(values,target,false,false,1);}catch(InvalidOperationException){Console.WriteLine("limited run: unknown");}
var completed=SubsetPlanning.FindSubset(values,target);
Console.WriteLine($"completed run: feasible={completed.Indices is not null}, calls={completed.Calls}, hits={completed.Hits}");
public static class SubsetPlanning
{
    public record SubsetResult(int[]? Indices,int Calls,int Hits,int Prunes,int FailedStates);
    // Signed values are allowed; positions are distinct and each is used once.
    public static SubsetResult FindSubset(int[] values,long target,bool prune=true,bool memoize=true,int maxCalls=1_000_000)
    {
        ArgumentNullException.ThrowIfNull(values);
        if(values.Length>128)throw new ArgumentException("At most 128 positions in this teaching implementation.");
        if(maxCalls<1)throw new ArgumentOutOfRangeException(nameof(maxCalls));
        int[] data=(int[])values.Clone();int n=data.Length;
        var lower=new long[n+1];var upper=new long[n+1];
        for(int i=n-1;i>=0;i--)
        {lower[i]=lower[i+1]+Math.Min(0,data[i]);upper[i]=upper[i+1]+Math.Max(0,data[i]);}
        var failed=new HashSet<(int Index,long Remaining)>();var chosen=new List<int>();
        int calls=0,hits=0,prunes=0;int[]? answer=null;
        bool Search(int index,long remaining)
        {
            if(calls==maxCalls)throw new InvalidOperationException("Search budget exhausted; feasibility is unknown.");
            calls++;
            if(remaining==0){answer=chosen.ToArray();return true;}
            if(index==n)return false;
            if(prune&&(remaining<lower[index]||remaining>upper[index])){prunes++;return false;}
            var state=(index,remaining);
            if(memoize&&failed.Contains(state)){hits++;return false;}
            chosen.Add(index);bool found;
            try{found=Search(index+1,remaining-data[index]);}
            finally{chosen.RemoveAt(chosen.Count-1);}
            if(found||Search(index+1,remaining))return true;
            if(memoize)failed.Add(state);
            return false;
        }
        // Even an unpruned trace rejects targets outside the total signed range.
        if(target<lower[0]||target>upper[0]){calls=1;prunes=1;}
        else Search(0,target);
        if(chosen.Count!=0)throw new InvalidOperationException("Undo failed.");
        return new SubsetResult(answer,calls,hits,prunes,failed.Count);
    }
}


limited run: unknown
completed run: feasible=False, calls=9, hits=1


## Submit

```{index} Submit
```

Submit your true/false explanations, code, outputs, state definitions, and witness checks through the assignment panel when available.